# ゼロから作るショアのアルゴリズム 第5回: すべてを組み合わせる

[第4回](ja/323_shor_scratch_modular_exponentiation.ipynb)ではべき剰余 $\ket{j}\ket{1} \to \ket{j}\ket{a^j \bmod N}$ を作りました — これがこのシリーズ最後の算術演算の部品です。今回はショアのアルゴリズムの残り3つの部品でそれを包み込み、このシリーズで初めて、アルゴリズム全体を(シミュレートされた)実際の量子ビット上で最初から最後まで実行します:

1. **位相推定の準備**: 指数レジスタを(第4回でテストのために行ったような古典的な値の設定ではなく)アダマールゲートによって等しい重ね合わせにします。
2. **逆量子フーリエ変換**をそのレジスタに適用し、べき剰余によって作られた周期的なもつれを、$1/r$ の倍数の直接測定可能な推定値に変換します。ここで $r$ は $a$ の $N$ を法とした位数です。
3. **古典的な連分数展開による後処理**で、その測定値を候補の位数 $r$ に変換し — $a^r \equiv 1 \pmod N$ を検証したうえで — $\gcd(a^{r/2}\pm 1, N)$ を通じて $N$ の因数を求めます。

理論的にはどれも新しいものではありません: [310_shor.ipynb](ja/310_shor.ipynb)がすでに、2つの具体的なケース($N=15$ と $N=21$)向けに手作りされた回路を使って、この3つのステップすべてを詳しく解説しています。ここで新しいのは、ステップ1・2が「このシリーズ自身が」作った、任意の $a$ と $N$ に対応するべき剰余回路(Part 1〜4で `X`/`CX`/`CCX` から積み上げたもの)の上で動く点です — 事前に最適化されたブラックボックスではありません。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995)

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

  Cloning https://github.com/blueqat/blueqatSDK to /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-uylkiopv


  Running command git clone --filter=blob:none --quiet https://github.com/blueqat/blueqatSDK /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-uylkiopv


  Resolved https://github.com/blueqat/blueqatSDK to commit 0fc0fa63291cbc1bb4303b97e22bdb7e76adf967


  Installing build dependencies ... -

 \

 done


  Getting requirements to build wheel ... -

 done


  Preparing metadata (pyproject.toml) ... -

 done


## おさらい: 第4回の算術演算

以下のセルは第4回からそのままコピーしたものです: 一般化された多重制御Xゲートのヘルパー `mcx`、`(controls, target)` の演算リストとしての加算器・剰余加算器の部品、そして制御乗算の中のその場でのスワップステップのための `controlled_swap_ops` です。それぞれの部品がどのように導出・検証されたかは第1〜4回を参照してください。

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls` (see Part 3 for the derivation)."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N -- the five-step construction from Part 2, as (controls, target) pairs."""
    ops = []
    ops += add_ops(c, a, b, n)
    ops += sub_ops(c, N, b, n)
    ops.append(((b[n],), t))
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]
    ops += sub_ops(c, a, b, n)
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def controlled_swap_ops(ctrl, a_reg, b_reg):
    """Swap a_reg[i] <-> b_reg[i] for every i, controlled by `ctrl` (Fredkin gate, built from CX/CCX/CX)."""
    ops = []
    for a, b in zip(a_reg, b_reg):
        ops.append(((b,), a))
        ops.append(((a, ctrl), b))
        ops.append(((b,), a))
    return ops

## 指数レジスタを重ね合わせにする

第4回の `build_modexp` は、`pow()` の出力と照合できるように、指数レジスタを `X` ゲートで古典的に設定していました。実際のアルゴリズムに必要な変更はただ一つ: その `X` ゲートを `H` ゲートに置き換えることです。1つの古典的な $j$ に対して正しく $a^j \bmod N$ を計算するのと全く同じ回路が、線形性により、レジスタが等しい重ね合わせから始まる場合には*すべての* $j$ について同時にそれを計算します。

それ以外 — レジスタの構成、指数の各ビットに対する制御乗算・制御スワップ・制御逆乗算のループ — は第4回とまったく同じです。

In [3]:
def build_modexp_qpe(mult_a, val_N, n, n_exp_bits, n_ancilla=4):
    """Same circuit as Part 4's build_modexp, but with the exponent register in superposition
    (Hadamards) instead of set classically (X gates) -- this is the circuit the real algorithm runs."""
    c = list(range(n))                                   # carry register
    const = list(range(n, 2 * n))                        # scratch: holds a constant, loaded/unloaded via X
    res = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide, becomes the new x)
    N = list(range(3 * n + 1, 4 * n + 1))                 # modulus
    x = list(range(4 * n + 1, 5 * n + 1))                 # the value being exponentiated
    j = list(range(5 * n + 1, 5 * n + 1 + n_exp_bits))    # exponent register: one control qubit per bit
    t = 5 * n + 1 + n_exp_bits                            # modulo-adder overflow flag
    ancillas = list(range(5 * n + 2 + n_exp_bits, 5 * n + 2 + n_exp_bits + n_ancilla))
    circ = Circuit(5 * n + 2 + n_exp_bits + n_ancilla)

    circ.x[x[0]]  # x starts at 1
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    for k in range(n_exp_bits):
        circ.h[j[k]]  # <-- the only change from Part 4: superposition, not a classical value

    def apply_modulo_add(ctrl, constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]
            for controls, target in ops_source(c, const, res, N, t, n):
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]

    for k in range(n_exp_bits):
        a_pow = pow(mult_a, 1 << k, val_N)      # a^(2^k) mod N, controlled by bit k of the exponent
        a_pow_inv = pow(a_pow, -1, val_N)
        apply_modulo_add(j[k], a_pow, modulo_add_ops)
        for controls, target in controlled_swap_ops(j[k], x, res[:n]):
            circ += mcx(list(controls), target, ancillas)
        apply_modulo_add(j[k], a_pow_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x, j

## 逆量子フーリエ変換

`build_modexp_qpe` の後、状態は(正規化を除いて)次のようになります:

$$\sum_{j=0}^{2^t-1} \ket{j}\ket{a^j \bmod N}$$

指数レジスタ `j` と結果レジスタ `x` がもつれた状態です。`j` に逆QFTをかけ、`j` だけを測定すると、[310_shor.ipynb](ja/310_shor.ipynb)が導出している通り、高い確率で $2^t/r$ の倍数の推定値が得られます。

QFT回路自体は標準的なもの(アダマール + 制御位相回転 + 最後にスワップによるビット反転)です。このシリーズ特有の一点: ビット `j[k]` は $a^{2^k}$ を制御するので、`j[-1]`(最後の量子ビット)が最大のべきを制御し、最も多くの位相情報を持ちます — そのためQFT回路からはレジスタの最上位ビットとして扱われる必要があり、これが以下の呼び出しで `j` を反転させて渡している理由です。

In [4]:
import math

def apply_qft_dagger(circuit, qubits):
    """Inverse QFT on `qubits`, with qubits[0] treated as the most-significant qubit."""
    num_qubits = len(qubits)
    for i in range(num_qubits // 2):
        circuit.swap(qubits[i], qubits[num_qubits - i - 1])
    for i in range(num_qubits):
        for k in range(i):
            circuit.cphase(-math.pi / (2 ** (i - k)))[qubits[k], qubits[i]]
        circuit.h[qubits[i]]

## 古典的な後処理: 連分数展開

指数レジスタの測定は、ある未知の $k$ について整数 $B \approx 2^t \cdot k/r$ を与えます。`fractions.Fraction(B, 2**t).limit_denominator(N)` は $k/r$ を既約分数として復元します — まさに[310_shor.ipynb](ja/310_shor.ipynb)が使っているのと同じトリックです — その分母が位数 $r$ の候補になります。

その候補は検証して初めて信頼できます: $a^r \equiv 1 \pmod N$ が実際に成り立つか確認する必要があります(検証していない $r$ に対して偶然のように見えるgcdの一致が出ても、それは証明にはなりません — 明示的に確認する価値があります。そうしないと簡単に騙されてしまいます)。検証できたら: $r$ が奇数であるか、$a^{r/2} \equiv -1 \pmod N$ である場合、この $a$ は「運が悪く」何の情報も得られません(これは*一部の* $a$ の選び方で起こることが期待される、正常な事態です)。そうでなければ、$\gcd(a^{r/2}-1, N)$ と $\gcd(a^{r/2}+1, N)$ を調べて $N$ の非自明な因数を探します。

In [5]:
import fractions
from collections import Counter
from math import gcd

def run_order_finding(mult_a, val_N, n, t_bits, shots=2000):
    circ, x, j = build_modexp_qpe(mult_a, val_N, n, t_bits)
    apply_qft_dagger(circ, list(reversed(j)))
    total = circ.n_qubits
    print(f"[qubits={total}, gates={len(circ.ops)}]")
    # NOTE: blueqatSDK's Circuit.run() defaults to mode="tensornet" (an opt_einsum-based
    # contraction), which is great for many circuits but pathologically slow for this one --
    # this circuit reuses ancilla/carry qubits heavily across thousands of gates, and the
    # tensor network opt_einsum ends up building is expensive to contract. Explicitly
    # requesting mode="statevector" (eager gate-by-gate simulation) instead is dramatically
    # faster here. If you have a CUDA GPU available, device="cuda" helps too.
    result = circ.m[:].run(shots=shots, mode="statevector")

    counts = Counter()
    for state, cnt in result.items():
        bits = "".join(state[total - 1 - q] for q in j)
        counts[int(bits, 2)] += cnt
    return counts

def guess_factor(mult_a, val_N, B, t_bits):
    r = fractions.Fraction(B, 2 ** t_bits).limit_denominator(val_N).denominator
    if pow(mult_a, r, val_N) != 1:
        return r, None, f"a^r = {pow(mult_a, r, val_N)} != 1 mod {val_N} -- r not verified, discard"
    if r % 2 != 0:
        return r, None, "r is odd -- no factor from this measurement"
    power = pow(mult_a, r // 2, val_N)
    f1, f2 = gcd(power - 1, val_N), gcd(power + 1, val_N)
    factors = [f for f in (f1, f2) if f not in (1, val_N)]
    if not factors:
        note = (f"a^(r/2) = {power} = -1 mod {val_N} -- " if power == val_N - 1 else "") + \
               "gcd gives only trivial factors from this measurement"
        return r, None, note
    return r, factors, None

## 実行してみる: $a=5, N=6$ の位数発見

$N=6=2\times3$ は正真正銘の合成数なので、これは単なる周期発見の健全性チェックではなく、本物の因数分解のデモです。この規模($n=3$ ビットの算術、$t=2$ の計数量子ビット — 合計23量子ビット)は、CPU1コアで2分弱で実行でき、[第3回](ja/322_shor_scratch_controlled_multiplier.ipynb)で議論したショットサンプリングの上限にも余裕を持って収まります。$5 \bmod 6$ の真の位数は $r=2$ です($5^2 = 25 \equiv 1$ のため)。$2^t=4$ が $r$ でちょうど割り切れるので、理論上は $B=0$ と $B=2$ にのみ鋭いピークが現れるはずです。

In [12]:
mult_a, val_N, n, t_bits = 5, 6, 3, 2
counts = run_order_finding(mult_a, val_N, n, t_bits, shots=2000)
total_shots = sum(counts.values())
for B in sorted(counts):
    print(f"B={B:2d}: {counts[B]:5d} shots ({100*counts[B]/total_shots:.1f}%)")

[qubits=23, gates=5838]
B= 0:   973 shots (48.6%)
B= 2:  1027 shots (51.4%)


In [7]:
for B in sorted(counts):
    r, factors, note = guess_factor(mult_a, val_N, B, t_bits)
    print(f"B={B}: guessed r={r}", f"-> factors {factors}" if factors else f"-> {note}")

B=0: guessed r=1 -> a^r = 5 != 1 mod 6 -- r not verified, discard
B=2: guessed r=2 -> factors [2]


$B=2$ は $r=2$(検証済みの真の位数: $5^2 \equiv 1 \pmod 6$)を正しく復元し、$\gcd(5^1-1, 6) = \gcd(4,6) = 2$ から本物の非自明な因数が得られます: **$6 = 2 \times 3$**。$B=0$ は自明な $r=1$ を与え、これは何の情報も持ちません — これは失敗ではなく、期待される結果の一つです。ショアのアルゴリズムは本質的に確率的であり、実際の実装では情報が得られない測定が出た場合は単に再試行します。

## 規模を拡大する: 教科書の定番例 $N=15$

$N=15=3\times5$(まさに[310_shor.ipynb](ja/310_shor.ipynb)が使っているのと同じ $N$)には $n=4$ ビットの算術が必要です。$a=7$、$t=2$ の計数量子ビットで実行すると、**28量子ビット、10,626個の高レベル演算**になります — これはラップトップのCPUで現実的に扱える範囲を超えていますが、メモリに余裕のあるGPU1枚であれば十分に実行可能です。NVIDIA H100(`device="cuda"`、`mode="statevector"`)で実行したところ、**58分**で完了し、4通りの測定値すべて($B=0,1,2,3$、それぞれ約25%)にわたって完全に均一な分布が得られました — これは期待通りの結果です($t=2$ のとき真の位数 $r=4$ がちょうど割り切れるため $2^t/r=1$ となり、すべての測定が有益な情報を持つことになります)。$a^r \equiv 1 \pmod{15}$ を確認して各候補の位数を検証すると:

| $B$ | 推定した $r$ | $a^r \bmod 15$ | 検証 | 因数 |
|---|---|---|---|---|
| 0 | 1 | 7 | 否 | — |
| 1 | **4** | 1 | 是 | **3, 5** |
| 2 | 2 | 4 | 否 | (gcdは偶然3を返すが、$r=2$ は未検証で信頼できない) |
| 3 | **4** | 1 | 是 | **3, 5** |

$B=1$ と $B=3$ は正しく真の位数 $r=4$ を復元し、$\gcd(7^2-1, 15) = \gcd(48,15) = 3$、$\gcd(7^2+1, 15) = \gcd(50,15) = 5$ から両方の素因数が得られます: **$15 = 3\times5$** — [310_shor.ipynb](ja/310_shor.ipynb)がその手作業で最適化された $N=15$ 専用回路で到達しているのと同じ結果を、このシリーズの完全に汎用的な、任意の $a$ と $N$ に対応する構成から得ています。($B=2$ の $\gcd=3$ はフラグを立てるべき偶然の一致であって、検証された結果ではありません — その $r=2$ は $a^r\equiv1$ のチェックに失敗しており、まさにこのチェックが重要である理由です。)

さらに本格的なケース — より高い解像度の位相推定のために計数量子ビットを増やした $N=15$ や、$N=21$ など — は、計算時間よりも手強い壁にぶつかります: `torch.multinomial`(blueqatのショットサンプリングが内部で使っている関数)には、利用可能なメモリ量とは無関係な、固定の$2^{24}$カテゴリという上限があります。それを超えると、ショットベースのサンプリングは単純にエラーになります。上記の $N=15$ の結果を得るために使った回避策は、ショットサンプリングを一切使わず、状態ベクトル全体から位相レジスタの各測定値の*厳密な*周辺確率を直接計算する(他のすべての量子ビットについて $|\text{振幅}|^2$ を足し合わせる)というものでした — `torch` のテンソルコードで数行の話ですが、このノートブックの焦点を量子回路自体に保つため、ここでは示していません。

## シリーズのまとめ

5回にわたって、このシリーズは `X`/`CX`/`CCX`/`H`/`cphase` だけからショアのアルゴリズムの量子部分の仕組みを作り上げてきました:

1. [第1回](ja/320_shor_scratch_adder.ipynb): 量子加算器
2. [第2回](ja/321_shor_scratch_modulo_adder.ipynb): 剰余加算
3. [第3回](ja/322_shor_scratch_controlled_multiplier.ipynb): 制御剰余乗算(そしてこの構成のコストの限界がどこにあるかについての考察)
4. [第4回](ja/323_shor_scratch_modular_exponentiation.ipynb): べき剰余。指数の各ビットにわたって制御乗算を連鎖させたもの
5. **第5回**(このノートブック): アダマール + べき剰余 + 逆QFT + 連分数展開を組み合わせて、実際に動く完全な位数発見アルゴリズムとして実行 — $N=6$ と $N=15$ の両方で実際の因数を復元

周囲の理論 — なぜ位数発見が因数分解を解くのか、そして手作業で最適化された回路を使った古典的な $N=15$・$N=21$ の実例については、このシリーズが実践的な伴走者であり続けてきた [310_shor.ipynb](ja/310_shor.ipynb) を参照してください。